# 3 — Regression: by hand, then with the tools

You know what a regression *is*. This notebook is about getting one out of a
computer, twice:

1. **With matrix algebra you write yourself** — `(X'X)⁻¹X'y`, the residuals, the
   standard errors, the t-statistics. About fifteen lines of NumPy.
2. **With `statsmodels`** — which does all of that in one line, and a great deal
   more that you should not write yourself.

Doing it by hand first is not nostalgia. When a package gives you a beta of 47 or
a standard error of `nan`, the people who can fix it are the ones who know what
the package was trying to compute.

**Assumed:** notebooks 1 and 2 — NumPy arrays, pandas, and enough plotting to
draw a line through a scatter.

**One new package:** `statsmodels`. If `import statsmodels` fails, run
`python -m pip install -r requirements.txt` from the project root.

In [ ]:
# Run me first. This makes workbook.py importable whatever folder Jupyter
# started in, then pulls in the three helpers you will use all the way through.
import sys
from pathlib import Path

for candidate in [Path.cwd(), Path.cwd() / 'Learn_To_Code', *Path.cwd().parents]:
    if (candidate / 'workbook.py').exists():
        sys.path.insert(0, str(candidate))
        break

from workbook import check, hint, todo, ensure_data, DATA_DIR

ensure_data()   # builds the workbook CSVs on your Desktop the first time only

print('Ready. Data lives in:', DATA_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.axisbelow': True})

print('statsmodels', sm.__version__)

## 1. The data, and the question

Twenty years of monthly returns: three market factors, a risk-free rate, a
recession flag, and three funds.

The data is **simulated**, which is deliberate. It means the true coefficients
exist and are written down — so when you fit a model you can check whether your
code recovered them. You cannot do that with real data, and it is the fastest way
to find out that your regression code is wrong.

In [ ]:
factors = pd.read_csv(DATA_DIR / 'factor_data.csv', parse_dates=['date'])
print(factors.shape)
factors.head()

### Fix the column names before you do anything else

`Mkt-RF` is the conventional Fama-French name, and it is unusable in code: the
hyphen is a minus sign to Python, so `df.Mkt-RF` is a subtraction and the formula
syntax you will meet in section 7 chokes on it.

Rename first. Lower case, underscores, no punctuation. Ten seconds now saves an
hour later.

In [ ]:
factors = factors.rename(columns={'Mkt-RF': 'mkt_rf', 'SMB': 'smb',
                                  'HML': 'hml', 'RF': 'rf'})
print(list(factors.columns))

### Excess returns

The factor model is about returns **in excess of** the risk-free rate — a fund
that earns 4% while cash earns 4% has produced nothing. The file gives total fund
returns, so subtract `rf` yourself. Forgetting this step is the most common error
in factor regressions, and it shows up as an alpha roughly equal to the average
short rate.

In [ ]:
factors['value_excess']  = factors['value_fund']  - factors['rf']
factors['growth_excess'] = factors['growth_fund'] - factors['rf']
factors['index_excess']  = factors['index_fund']  - factors['rf']

factors[['mkt_rf', 'smb', 'hml', 'rf', 'value_excess']].describe().round(4)

**The question for sections 2 to 7:** value_fund returned more than the market.
Was that skill, or was it exposure to things that happened to do well?

In [ ]:
fig, ax = plt.subplots()
ax.scatter(factors['mkt_rf'] * 100, factors['value_excess'] * 100,
           s=18, alpha=0.7)
ax.set_xlabel('Market excess return (% per month)')
ax.set_ylabel('Fund excess return (% per month)')
ax.set_title('One dot per month, 240 months')
plt.show()

## 2. Getting the data into matrix shape

Everything that follows is one formula:

$$\hat\beta = (X'X)^{-1}X'y$$

To use it you need `y` and `X` as arrays with the right shapes:

* `y` — the dependent variable, shape `(n,)`, one entry per observation.
* `X` — the design matrix, shape `(n, k)`, one **column per coefficient**.

The constant is not special. It is a column of ones. If you leave it out you are
forcing the line through the origin, and every other coefficient becomes wrong to
compensate.

Two functions do the work:

```python
y = df['col'].to_numpy()                       # Series -> 1-D array
X = np.column_stack([np.ones(n), df['x']])     # glue columns side by side
```

**Check `.shape` every single time.** Nearly every regression bug is a shape bug,
and a `(240, 1)` where a `(240,)` belongs will silently broadcast into a
240×240 matrix and eat your memory.

In [ ]:
y = factors['value_excess'].to_numpy()
n = len(y)

X = np.column_stack([np.ones(n), factors['mkt_rf'].to_numpy()])

print('y', y.shape, '| X', X.shape)
print(X[:4])

### Your turn

In [ ]:
# Build the design matrix for the market model yourself:
# a column of ones, then the market excess return. Shape should be (240, 2).

X = todo()

print(X.shape)
check('3.1', X)

## 3. The coefficients

The literal translation of the formula uses `np.linalg.inv`:

```python
b = np.linalg.inv(X.T @ X) @ X.T @ y
```

`@` is matrix multiplication and `.T` is the transpose. Read it right to left:
`X.T @ y` is a `(k,)` vector, `X.T @ X` is `(k, k)`, and the inverse of that maps
one into the other.

It works. You should still not write it that way.

In [ ]:
X = np.column_stack([np.ones(n), factors['mkt_rf'].to_numpy()])

b_inv  = np.linalg.inv(X.T @ X) @ X.T @ y     # the literal formula
b_hand = np.linalg.solve(X.T @ X, X.T @ y)    # what you should actually write

print('inv  ', b_inv)
print('solve', b_hand)
print('largest difference:', np.abs(b_inv - b_hand).max())

**Why `solve`.** Asking for an inverse and then multiplying is more arithmetic
than the problem needs, and every extra operation loses a little precision. When
the columns of `X` are strongly correlated — which happens constantly with
financial regressors — that lost precision becomes visible. `np.linalg.solve`
answers the question "which `b` satisfies `(X'X)b = X'y`" directly.

A third option, `np.linalg.lstsq(X, y, rcond=None)`, solves the least-squares
problem without forming `X'X` at all. It is the most numerically stable of the
three and it still returns an answer when `X` is rank-deficient. Use `solve` by
default; reach for `lstsq` when `solve` complains.

**The rule of thumb:** if you are about to write `inv(A) @ b`, write
`solve(A, b)` instead. This is true everywhere in numerical linear algebra, not
just here.

### Your turn

In [ ]:
# Estimate the coefficients with np.linalg.solve.
# Two numbers come back: the intercept (alpha) and the slope (beta).

b = todo()

print(f'alpha {b[0]: .5f} per month   ({b[0] * 12: .2%} per year)')
print(f'beta  {b[1]: .4f}')
check('3.2', b)

A beta near 1.02 and an alpha of about 0.24% a month — roughly 2.9% a year. That
looks like a fund manager worth paying. Hold that thought until section 7.

## 4. Fitted values, residuals, R²

* **fitted** `ŷ = Xb` — what the model says each month should have been;
* **residual** `e = y − ŷ` — what it missed by;
* **R²** — the share of the variance of `y` the model accounts for:

$$R^2 = 1 - \frac{e'e}{(y-\bar y)'(y-\bar y)}$$

`e @ e` is the sum of squared residuals: in NumPy the `@` of two 1-D arrays is a
dot product, so it is the tidiest way to write `sum(e**2)`.

In [ ]:
fitted_hand = X @ b_hand
resid_hand  = y - fitted_hand

ss_res = resid_hand @ resid_hand
ss_tot = (y - y.mean()) @ (y - y.mean())
r2_hand = 1 - ss_res / ss_tot

print(f'R-squared      {r2_hand:.4f}')
print(f'residual mean  {resid_hand.mean():.2e}  (exactly 0 when X has a constant)')
print(f'corr(e, x)     {np.corrcoef(resid_hand, X[:, 1])[0, 1]:.2e}  (also 0, by construction)')

Those last two lines are not a coincidence and they are worth remembering: OLS
picks `b` precisely so that the residuals are orthogonal to every column of `X`.
If you ever compute residuals that correlate with a regressor, you have a bug.

A residual plot is the cheapest diagnostic there is — you are looking for a
shapeless cloud with no pattern:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(factors['mkt_rf'] * 100, y * 100, s=16, alpha=0.6)
grid = np.linspace(factors['mkt_rf'].min(), factors['mkt_rf'].max(), 50)
axes[0].plot(grid * 100, (b_hand[0] + b_hand[1] * grid) * 100,
             color='crimson', linewidth=2)
axes[0].set_xlabel('Market excess return (%)')
axes[0].set_ylabel('Fund excess return (%)')
axes[0].set_title('Fit')

axes[1].scatter(fitted_hand * 100, resid_hand * 100, s=16, alpha=0.6)
axes[1].axhline(0, color='crimson', linewidth=1.5)
axes[1].set_xlabel('Fitted value (%)')
axes[1].set_ylabel('Residual (%)')
axes[1].set_title('Residuals — you want no pattern here')

fig.tight_layout()
plt.show()

### Your turn

In [ ]:
# Compute the residuals and the R-squared for the market model.
# Use b from exercise 3.2 (or b_hand, the worked version above).

resid = todo()
r2    = todo()

print(f'R2 = {r2:.4f}')
check('3.3', resid, r2)

## 5. Standard errors and t-statistics

A coefficient without a standard error is not a result. Three steps:

$$s^2 = \frac{e'e}{n-k} \qquad
\widehat{\mathrm{Var}}(b) = s^2 (X'X)^{-1} \qquad
\mathrm{se}(b_j) = \sqrt{\widehat{\mathrm{Var}}(b)_{jj}}$$

Two details that trip people up:

* the denominator is **n − k**, not n — k is the number of coefficients, the
  constant included. Divide by n and every standard error comes out too small;
* the standard errors are the **square roots of the diagonal** of the covariance
  matrix. `np.diag` pulls that diagonal out.

Then `t = b / se`, and the p-value is the two-sided tail of a t distribution with
n − k degrees of freedom.

In [ ]:
# Rebuilt from scratch so this cell stands on its own — a habit worth keeping,
# because it means restarting the kernel and running one cell always works.
y = factors['value_excess'].to_numpy()
X = np.column_stack([np.ones(len(y)), factors['mkt_rf'].to_numpy()])
n, k = X.shape

b_hand = np.linalg.solve(X.T @ X, X.T @ y)
e_hand = y - X @ b_hand

s2 = (e_hand @ e_hand) / (n - k)       # residual variance
var_b = s2 * np.linalg.inv(X.T @ X)    # k x k covariance matrix of b
se_hand = np.sqrt(np.diag(var_b))

t_hand = b_hand / se_hand
p_hand = 2 * stats.t.sf(np.abs(t_hand), df=n - k)

results = pd.DataFrame({
    'coefficient': b_hand,
    'std error': se_hand,
    't-stat': t_hand,
    'p-value': p_hand,
}, index=['alpha', 'beta_mkt'])

results.round(4)

This is the little table you will build a hundred times. Notice `inv` is fine
here — you genuinely need the whole inverse matrix, not the solution of a system.

**Reading it:** beta is 1.02 with a t of 23 — the fund tracks the market, no doubt
about it. Alpha is 0.0024 with a t of 1.4, which does not clear the usual bar of
about 2. So the headline "2.9% a year of alpha" already comes with the caveat
that we cannot distinguish it from zero.

### Your turn

In [ ]:
# Compute the standard errors and the t-statistics for the market model.
# Remember: n - k, np.diag, and the square root.
# Use your own residuals from 3.2/3.3, or b_hand and e_hand from the cell above.

se      = todo()
t_stats = todo()

print(np.round(se, 5), np.round(t_stats, 2))
check('3.4', se, t_stats)

## 6. Wrap it up

You now have the whole calculation scattered over six cells. That is fine while
you are learning it and useless afterwards. Put it in a function: one place to
fix, works on any `y` and `X`, and you can trust it.

This is exactly the "write a function the second time you need it" rule from
notebook 1, applied to something real.

### Your turn

In [ ]:
def ols(y, X):
    """Ordinary least squares.

    Parameters
    ----------
    y : ndarray, shape (n,)
    X : ndarray, shape (n, k)   — include the constant column yourself

    Returns
    -------
    dict with keys 'beta', 'se', 'tstat', 'r2'
    """
    # your code here
    return {}


out = ols(y, X)
print(out)
check('3.5', ols)

The checker runs your function on a **fresh, randomly generated** dataset, not on
this one. If anything is hard-coded it will fail — which is the point of testing
a function rather than eyeballing its output once.

## 7. statsmodels

Everything above, in one line. Two things to know before you type it:

* **`y` comes first**, then `X` — the opposite of `scikit-learn`, and a classic
  source of nonsense results;
* **statsmodels does not add a constant for you.** `sm.add_constant(X)` does it.
  Forget it and you get a regression through the origin, with a beta that looks
  plausible and is wrong.

In [ ]:
mkt = factors['mkt_rf'].to_numpy()

model = sm.OLS(y, sm.add_constant(mkt))
fit = model.fit()

print('statsmodels coefficients :', np.round(fit.params, 6))
print('by hand                  :', np.round(b_hand, 6))
print('standard errors match    :', np.allclose(fit.bse, se_hand))
print('R-squared matches        :', np.isclose(fit.rsquared, r2_hand))

Identical. That is the payoff of section 3 to 5: the package is not a black box
any more, it is your fifteen lines written by somebody careful.

### The summary table

In [ ]:
print(fit.summary())

There is a lot there. In practice you read five things:

| Where | What | This model |
| ----- | ---- | ---------- |
| `coef` | the estimates | alpha 0.0024, beta 1.02 |
| `std err` / `t` / `P>|t|` | is it distinguishable from zero? | beta yes, alpha no |
| `R-squared` | share of variance explained | 0.69 |
| `No. Observations` | did you lose rows to NaNs? | 240 — nothing dropped |
| `Durbin-Watson`, `Prob(Omnibus)` | autocorrelation and non-normal residuals | flags, not verdicts |

Everything on the fitted object is reachable in code, which is what you want when
you are running two hundred regressions rather than reading one:

```python
fit.params      fit.bse        fit.tvalues    fit.pvalues
fit.rsquared    fit.rsquared_adj              fit.resid
fit.fittedvalues               fit.nobs       fit.conf_int()
```

### Your turn

In [ ]:
# Fit the same market model with statsmodels. Do not forget the constant.

result = todo()

print(result.params)
check('3.6', result)

## 8. The formula API — where it gets pleasant

`statsmodels.formula.api` takes an R-style formula and a DataFrame, and builds
the design matrix for you:

```python
smf.ols('y_column ~ x1 + x2', data=df).fit()
```

* the intercept is included automatically (`- 1` removes it);
* columns are referred to by name, so no `to_numpy()`, no `column_stack`, and no
  chance of lining `y` up against the wrong rows;
* rows with missing values are dropped for you — check `No. Observations` to see
  how many.

This is why we renamed `Mkt-RF` in section 1. Formulas need valid Python names.

In [ ]:
capm = smf.ols('value_excess ~ mkt_rf', data=factors).fit()
print(capm.params.round(5).to_string())
print(f'R2 {capm.rsquared:.4f}')

Now answer the question this notebook started with. Alpha survives the CAPM. Does
it survive controlling for size and value exposure?

In [ ]:
ff3 = smf.ols('value_excess ~ mkt_rf + smb + hml', data=factors).fit()

comparison = pd.DataFrame({
    'CAPM': capm.params,
    'CAPM t': capm.tvalues,
    'FF3': ff3.params,
    'FF3 t': ff3.tvalues,
}).reindex(['Intercept', 'mkt_rf', 'smb', 'hml'])       # NaN = not in that model

print(comparison.round(4).to_string())
print(f'\nR2: CAPM {capm.rsquared:.3f}  ->  FF3 {ff3.rsquared:.3f}')

**There is the answer.** The alpha falls from 0.0024 a month (2.9% a year) to
0.0007 (0.8% a year), and its t-statistic goes from 1.4 to 0.5. The fund loads
hard on size (0.30) and value (0.60), and those loadings explain the returns the
CAPM was calling alpha. It is a value fund, not a skilled one.

### What the data actually was

This is the moment simulated data earns its place. The true parameters used to
generate the file, per month:

| fund | alpha | mkt | smb | hml | extra market beta in recessions |
| ---- | ----- | --- | --- | --- | ------------------------------- |
| `growth_fund` |  0.0020 | 0.90 | −0.20 | −0.40 | **+0.50** |
| `value_fund`  |  0.0000 | 1.05 |  0.30 |  0.60 | 0 |
| `index_fund`  | −0.0002 | 1.00 |  0.02 |  0.00 | 0 |

Your FF3 estimates for `value_fund`: alpha ≈ 0.0007 against a truth of 0, beta
1.043 against 1.05, smb 0.304 against 0.30, hml 0.602 against 0.60. The code is
right, and the CAPM alpha was an artefact of leaving two factors out.

### More formula syntax

In [ ]:
# Interactions and transformations, without touching the DataFrame:
#   a * b   ->  a + b + a:b        (both terms and the interaction)
#   a : b   ->  just the interaction
#   C(x)    ->  treat x as categorical and make dummies
#   np.log(x), I(x ** 2)  ->  transform inline
#   - 1     ->  no intercept

print(smf.ols('value_excess ~ mkt_rf + C(recession)', data=factors)
      .fit().params.round(4).to_string())

### Your turn

In [ ]:
# Fit the three-factor model for value_excess using the formula API.

ff3_model = todo()

print(ff3_model.params.round(4).to_string())
check('3.7', ff3_model)

## 9. Standard errors that survive contact with financial data

Textbook OLS standard errors assume the residuals all have the same variance and
are uncorrelated across time. Financial data breaks both: volatility clusters, and
monthly series are autocorrelated.

The fix is not to change the estimator — **the coefficients do not move** — but to
compute the standard errors differently:

| `cov_type` | Handles | Use when |
| ---------- | ------- | -------- |
| default | nothing | rarely honest with real data |
| `'HC1'` | heteroskedasticity | cross-sections |
| `'HAC'` with `maxlags` | heteroskedasticity **and** autocorrelation | time series — this is the one for return data |

`maxlags` is a judgement call. A common starting point is about `4(n/100)^(2/9)`,
which for 240 months is 5 to 6.

In [ ]:
plain = smf.ols('value_excess ~ mkt_rf + smb + hml', data=factors).fit()
white = plain.get_robustcov_results(cov_type='HC1')
hac   = plain.get_robustcov_results(cov_type='HAC', maxlags=6)

se_table = pd.DataFrame({
    'coefficient': plain.params,
    'se (OLS)': plain.bse,
    'se (HC1)': white.bse,
    'se (HAC)': hac.bse,
}, index=plain.params.index)

print(se_table.round(5).to_string())
print('\ncoefficients identical:', np.allclose(plain.params, hac.params))

Same coefficients, different standard errors — here HAC is *smaller* for the
factor loadings and slightly larger for the intercept. That direction is not
guaranteed; the point is that inference changes and estimates do not.

You can also ask for the robust version when you fit:

```python
smf.ols('y ~ x', data=df).fit(cov_type='HAC', cov_kwds={'maxlags': 6})
```

### Your turn

In [ ]:
# Fit the three-factor model twice: once plain, once with HAC standard errors
# and maxlags=6. Use the fit(cov_type=..., cov_kwds=...) form.

plain_fit = todo()
hac_fit   = todo()

check('3.8', plain_fit, hac_fit)

## 10. Dummies and interactions

`recession` is 1 in recession months and 0 otherwise. Two very different things
you can do with it:

```python
y ~ mkt_rf + recession        # recessions shift the line up or down
y ~ mkt_rf * recession        # recessions change the SLOPE as well
```

The second expands to `mkt_rf + recession + mkt_rf:recession`, and that
interaction term is the interesting one: it is how much the market beta *changes*
in recessions.

Try it on `growth_fund`, which the CAPM says has a beta of about 0.97.

In [ ]:
simple = smf.ols('growth_excess ~ mkt_rf + smb + hml', data=factors).fit()
inter  = smf.ols('growth_excess ~ mkt_rf * recession + smb + hml', data=factors).fit()

print('one beta for all months:')
print(f"  mkt_rf {simple.params['mkt_rf']:.3f}   R2 {simple.rsquared:.4f}\n")

print('beta allowed to differ in recessions:')
print(inter.params.round(4).to_string())
print(f"\n  R2 {inter.rsquared:.4f}")

normal = inter.params['mkt_rf']
recess = inter.params['mkt_rf'] + inter.params['mkt_rf:recession']
print(f"\n  beta in normal months    {normal:.3f}")
print(f"  beta in recessions       {recess:.3f}   (t on the interaction: "
      f"{inter.tvalues['mkt_rf:recession']:.2f})")

The single-beta model says 0.97 and hides the whole story. Allowing the slope to
change says the fund runs a beta of 0.91 in calm months and 1.43 when the market
falls — it takes on *more* market risk exactly when that hurts most. The true
value was 0.90 rising to 1.40, so the interaction recovered it.

That is a conclusion no amount of staring at a scatter plot would have given you,
and it comes from one `*` in a formula.

**Careful with the interaction coefficient.** `mkt_rf:recession` is 0.52, and 0.52
is not the recession beta. It is the *difference*. You have to add the base term.

### Your turn

In [ ]:
# Fit growth_excess on mkt_rf interacted with recession (plus smb and hml),
# then work out the market beta that applies during recessions.

model_int        = todo()
beta_in_recession = todo()

print(f'beta in recessions: {beta_in_recession:.3f}')
check('3.9', model_int, beta_in_recession)

## 11. Three ways this goes wrong

### Perfect collinearity

If one column of `X` is an exact combination of the others, `X'X` has no inverse
and there is no unique answer. This happens most often by accident: adding a
dummy for *every* category as well as a constant, or including both a variable
and a rescaling of it.

In [ ]:
X_bad = np.column_stack([np.ones(n), mkt, mkt * 2.0])   # third column is redundant

try:
    np.linalg.solve(X_bad.T @ X_bad, X_bad.T @ y)
except np.linalg.LinAlgError as exc:
    print('NumPy refuses:', exc)

bad_fit = sm.OLS(y, X_bad).fit()
print('\nstatsmodels does not stop — it warns and uses a pseudo-inverse:')
print(np.round(bad_fit.params, 4))
print('condition number:', f'{bad_fit.condition_number:.3e}')

NumPy raising is the *friendly* behaviour: it tells you immediately. statsmodels
warns and then hands back numbers anyway, and those numbers are meaningless — the
split between the two collinear columns is arbitrary, so `0.20` and `0.41` could
equally well have been `1.02` and `0.00`.

**A condition number above about 1e3 means go and look at your design matrix.**
`summary()` prints a note about it at the bottom for exactly this reason.

### Silently dropped rows

The formula API drops rows with missing values without telling you. Always
compare `No. Observations` with the size of your data.

In [ ]:
messy = factors.copy()
messy.loc[messy.index[:40], 'smb'] = np.nan

short = smf.ols('value_excess ~ mkt_rf + smb', data=messy).fit()
print(f'rows in the data : {len(messy)}')
print(f'rows in the model: {int(short.nobs)}   <- 40 months quietly gone')

### Regressing on the future

`smb` explains `value_excess` *in the same month*. That is a decomposition, not a
forecast, and it is a perfectly good thing to do — as long as you do not describe
it as prediction. If you want to forecast, the right-hand side has to be shifted:

```python
df['mkt_lagged'] = df['mkt_rf'].shift(1)    # last month's market
```

`shift(1)` moves the data *down*, so row t holds the value from t−1. `shift(-1)`
goes the other way and is how look-ahead bias gets into a backtest.

## 12. Capstone: a rolling beta

One regression on twenty years assumes the relationship never changed. A rolling
regression drops that assumption: fit the model on months 1–36, then 2–37, then
3–38, and watch the coefficient move.

The recipe:

```python
for i in range(window, len(df) + 1):
    chunk = df.iloc[i - window:i]        # the last `window` rows ending at i
    ...fit...
    betas.append(slope)
```

`len(df) + 1` in the `range` is not a typo — it is what makes the final window end
on the last row. With 240 months and a 36-month window you should get 205 betas.

### Your turn

In [ ]:
window = 36

# Rolling market beta of growth_excess on mkt_rf.
# Build a list of slopes, one per window, then turn it into an array.

betas = []

# your loop here

betas = np.array(betas)
print(len(betas), 'betas')
check('3.10', betas)

In [ ]:
# Once the check passes, this draws it. The dates are the END of each window.
if len(betas) == len(factors) - window + 1:
    dates = factors['date'].iloc[window - 1:]

    fig, ax = plt.subplots()
    ax.plot(dates, betas, linewidth=1.8)
    ax.axhline(0.90, color='0.5', linestyle='--', linewidth=1,
               label='true calm-market beta (0.90)')

    in_rec = factors['recession'] == 1
    for d in factors.loc[in_rec, 'date']:
        ax.axvspan(d - pd.Timedelta(days=15), d + pd.Timedelta(days=15),
                   color='0.85', zorder=0)

    ax.set_xlabel('End of 36-month window')
    ax.set_ylabel('Market beta')
    ax.set_title('Beta is not a constant (grey = recession months)')
    ax.legend()
    plt.show()
else:
    print('finish exercise 3.10 first')

Two things to take from that chart.

First, **beta is not a constant.** It sits near 0.85 in calm stretches and climbs
past 1.2 around the recessions — the same result section 10 got from one
interaction term, now visible over time.

Second, and less obvious: **beta stays high for three years after each recession
ends.** Nothing changed in the fund. The 36-month window still contains those
months, and it keeps containing them until it has rolled past. Every rolling
estimate lags reality by roughly its own window length, and reading that lag as a
real change is a mistake people make constantly. Shorter windows react faster and
are noisier; that trade-off is the whole design decision.

## 13. What about scikit-learn?

`sklearn.linear_model.LinearRegression` fits the same line and is the right tool
when you care about **prediction**: it is built for cross-validation, pipelines,
regularisation, and models with a thousand features.

It deliberately gives you no standard errors, no t-statistics and no p-values,
because those are not what it is for. For a question like "is this alpha
distinguishable from zero", use statsmodels. Two tools, two jobs — and it is not
in this project's requirements because you do not need it here.

## 14. Where you are

| | |
| --- | --- |
| **by hand** | design matrix, `solve` over `inv`, residuals, R², `s²(X'X)⁻¹`, t-stats |
| **statsmodels** | `sm.OLS`, `add_constant`, `.summary()`, and everything on the fitted object |
| **formulas** | `smf.ols('y ~ a + b')`, `C()`, `*` interactions, inline transforms |
| **inference** | HC1 and HAC standard errors, and why the coefficients never move |
| **structure** | dummies, interactions, rolling windows |
| **failure** | collinearity, dropped rows, look-ahead |

### Try it on real data

Everything here works unchanged on the live Fama-French factors:

```python
import sys
from workbook import PROJECT_ROOT
sys.path.insert(0, str(PROJECT_ROOT))

from Data.data_definition import DataDefinition

ff = DataDefinition(source='famafrench',
                    item='F-F_Research_Data_Factors',
                    start='1990-01-01', end=None).extract() / 100.0   # percent!
```

That `/ 100.0` is not optional — Kenneth French's files are in percentage points,
and forgetting it makes every annualised number come out about a hundred times
too big. The main README says more.

### Next

**Notebook 4 — Monte Carlo.** Regression fits a model to data you have.
Simulation generates data you do not, which is how you answer questions where
there is no sample to regress.